#<img src="https://img.stackshare.io/service/10345/ADB.png"/></a> Workshop Lab 3 - Workflow and Governance#
**Content**
* Orchestration
  * Job (Workflow)
* Governance
  * Unity Catalog  
  * Managed vs external tables
* SQL Warehouse
* AI  
  * Genie - AI assisted development
* Other (DAB, GIT, Market Lookup)
  

## 🔄 What is Orchestration?

**Orchestration** is the automated coordination of data and compute tasks into an end-to-end workflow. Instead of running each notebook, script, or pipeline **manually**, you define a workflow that the platform executes for you — in the right order, on the right compute, with retries and alerts if something fails.

### The task of orchestration

An orchestrator is responsible for:

* **Sequencing** – running tasks in the correct order (dependencies, e.g. "only load after the extract succeeded")
* **Scheduling** – triggering runs on a cron schedule, on file arrival, or on table updates
* **Compute management** – starting/stopping the clusters or warehouses each task needs
* **Error handling** – retries, failure notifications, and email/webhook alerts
* **Parameterization** – passing values (dates, paths, environment config) between tasks
* **Monitoring & observability** – run history, duration, and lineage for every execution

### How it relates to a Pipeline

Think of it as a two-level model:

| Concept | Role | Example
| --- | --- | --- |
| **Pipeline (SDP)** | *Processes data* — a set of transformations (bronze → silver → gold) running as one unit | A Lakeflow Spark Declarative Pipeline that cleans and aggregates data |
| **Job (Orchestration)** | *Coordinates everything around it* — schedules and chains tasks, including whole pipelines | A nightly job: ingest → run pipeline → refresh dashboard → notify |

So a **Pipeline is a task type inside a Job**. The Job decides *when* and *in what order* things run; the Pipeline decides *how the data is transformed*. Together they form a production-ready data workflow.

## 🎯 Challenge: Create a simple Workflow Job

In this exercise, you will build your first **Databricks Workflow Job** — turning customer pipeline into an automated, repeatable workflow.

### What you'll build

A Job that runs the ingestion notebook and customer pipeline as **sequential tasks** on a schedule:

1. **Task 1 – Load** – run the ingestion notebook (data_soruce_customer)
2. **Task 2 – Transform** – run the pipeline what created the previous lab (depends on Task 1)
3. **Task 3 – Notify** – send a success notification (depends on Task 2)

### Requirements

- The Job must run **on demand** and on a **cron schedule** (e.g. `0 6 * * *` — daily at 6 AM UTC)
- Each task should run on **serverless compute** or the shared cluster
- Configure **one retry** per task and an email alert on failure

### How to get started

1. In the workspace sidebar, click **Workflows** → **Create Job**
2. Add the notebook and pipeline tasks above, wiring the **Run task after** dependencies
3. Set the schedule, retries, and notifications under **Job settings**
4. Click **Run now** to test, then inspect the **Run history** and task-level logs

> 💡 **Tip:** Once it works, compare it to the Pipeline from the previous section — notice how the Job handles *scheduling and sequencing* while the tasks themselves do the data processing.

## 🛡️ Governance — Unity Catalog

### What is it?

**Unity Catalog (UC)** is Databricks' unified, built-in **data governance layer** — one central place to secure, govern, and discover all data and AI assets (tables, files, views, models, volumes, functions) across workspaces and clouds.

> 💡 **Why it matters:** without UC, every workspace has its own isolated metastore and permissions are scattered. UC gives you **one metadata layer for the entire organization** — security, lineage, auditing, and discovery in one place.

### The 3-level namespace

```
CATALOG  →  SCHEMA  →  TABLE / VIEW / VOLUME / MODEL / FUNCTION
```

| Level | Purpose | Example |
| --- | --- | --- |
| **Catalog** | Top-level domain, often maps to an environment or business area | `sales_prod` |
| **Schema** | Groups related objects (like a folder / database) | `sales_prod.crm` |
| **Object** | The actual asset users query | `sales_prod.crm.customers` |

Each level can have its own owner, grants, and tags — so access is naturally hierarchical.

### Most important governance capabilities

| Capability | What it gives you |
| --- | --- |
| **Schemas** | Logical grouping of data assets inside a catalog — keeps catalogs clean and modular |
| **Descriptions & comments** | Every catalog/schema/table/column can carry a description — powers **search & discovery** in Catalog Explorer |
| **Access control (grants)** | `GRANT SELECT ON TABLE ... TO group` — plus **row filters** and **column masks** to hide sensitive rows/cells from certain users |
| **Lineage** | Automatic, table- and column-level **data lineage** captured as data moves through pipelines — see exactly where a report's numbers come from |
| **Data quality monitoring** | **Labs / expectations** on pipelines, plus **monitoring** that profiles tables and alerts on drift, anomalies, and freshness over time |
| **Audit logs** | `system.access.audit` system tables log **who accessed what, when** — required for compliance (GDPR, HIPAA…) |
| **Tags (ABAC)** | Attach tags to objects for attribute-based governance and cost/PII classification |
| **Credentials & external locations** | Centralized access to cloud storage — no more pasted keys in notebooks |
| **Delta Sharing** | Share live data **across organizations** (even non-Databricks users) with open protocols — governed by the same UC grants |
| **Table constraints** | Primary key / foreign key constraints and CHECK rules enforce integrity at the catalog level |
| **AI governance** | Models, functions, and vector indexes live in UC too — so your **AI assets get the same governance** as tables |


### Managed vs external tables

| | **Managed table** | **External table** |
| --- | --- | --- |
| **Storage** | UC controls the location in your cloud account | You control the location (external location / path) |
| **DROP behavior** | Data is **deleted** | Only metadata removed — **files stay** |
| **Governance & performance** | Full UC features, liquid clustering, optimizations | Same governance, but storage lifecycle is yours |

> Default best practice: use **managed tables** unless the data must be shared or written by other tools.

### How you use it day-to-day

```sql
-- Grant access
GRANT USE CATALOG ON CATALOG sales_prod TO `group`;
GRANT USE SCHEMA, SELECT ON SCHEMA sales_prod.crm TO `group`;

-- Describe an asset for discovery
COMMENT ON TABLE sales_prod.crm.customers IS 'Customer master data, SCD2';

-- Create a managed table (no LOCATION → UC manages it)
CREATE TABLE sales_prod.crm.customers (...);
```

Explore everything in **Catalog Explorer** (sidebar → Catalog): search, lineage graph, grants, quality metrics, and audit history — all in one UI.

---

**TL;DR:** Unity Catalog = *one* place to define who can access which data (access), where it came from (lineage), what it means (descriptions/tags), whether it's good (data quality), and who did what (audit) — across the whole lakehouse.

%md

## 🏢 SQL Warehouse — compute for analytics

### What is it?

A **SQL Warehouse** (formerly SQL Endpoint) is Databricks' compute engine optimized for **SQL analytics** — dashboards, BI tools, ad-hoc queries, and SQL notebooks. Instead of a Spark cluster, you get a **serverless, massively parallel SQL engine** that starts in seconds and scales automatically.

### SQL Warehouse vs. "normal" (all-purpose cluster) compute

| | **SQL Warehouse** | **All-purpose compute (cluster)** |
| --- | --- | --- |
| **Best for** | SQL analytics, BI, dashboards, querying the lakehouse | Data engineering, notebooks, Spark jobs, pipelines, ML |
| **Workload type** | SQL only (Databricks SQL) | Spark (PySpark, Scala, SQL, R) |
| **Startup** | Serverless — starts in seconds, scale to zero | Minutes to spin up a cluster (or serverless compute) |
| **Scaling** | Automatic, independent scaling out for concurrency | Manual or autoscaling of fixed cluster size |
| **Optimization** | Cached results, cost-based optimizer tuned for analytics | General-purpose Spark tuning |
| **Cost** | Priced per DBU while running (scale to zero available) | Priced per DBU while running |
| **Access** | BI tools (Power BI, Tableau), SQL editor, alerts, queries | Notebooks, jobs, pipelines |

> 💡 **Rule of thumb:** analysts and BI tools → **SQL Warehouse**; engineers running pipelines, ingestion, or ML → **all-purpose / job compute**.

---

## 🧪 Challenge: Query your silver and gold tables

Time to use the tables you built in the previous labs! Open the **SQL Editor**, pick your SQL Warehouse, and complete the following exercises against your `silver` and `gold` tables.

### Exercise 1 — Inspect the layers

List the tables in your catalog and confirm what exists at each layer:

```sql
SHOW TABLES IN <your_catalog>.silver;
SHOW TABLES IN <your_catalog>.gold;
```

Then describe the structure of one gold table:

```sql
DESCRIBE TABLE <your_catalog>.gold.<your_gold_table>;
```

### Exercise 2 — Compare silver vs. gold

The **silver** layer holds cleaned, validated data; the **gold** layer holds business-level aggregates. Explore the difference:

```sql
-- Row counts: silver should be richer (row-level), gold more compact (aggregated)
SELECT COUNT(*) AS silver_rows FROM <your_catalog>.silver.<your_silver_table>;
SELECT COUNT(*) AS gold_rows FROM <your_catalog>.gold.<your_gold_table>;
```

Then compare a sample of rows from each layer and note what changed (granularity, columns, aggregations):

```sql
SELECT * FROM <your_catalog>.silver.<your_silver_table> LIMIT 10;
SELECT * FROM <your_catalog>.gold.<your_gold_table> LIMIT 10;
```

### Exercise 3 — Build your own insight

Write a query answering a business question using the **gold** table — for example, top N customers by revenue, monthly trend, or category breakdown:

```sql
-- Example pattern: top 10 by your gold table's key metric
SELECT *
FROM <your_catalog>.gold.<your_gold_table>
ORDER BY <metric_column> DESC
LIMIT 10;
```

### Exercise 4 — Prove the lineage (bonus)

Open **Catalog Explorer**, find your gold table, and click the **Lineage** tab. You should see it connected back to the silver table (and beyond to the source). This is Unity Catalog automatically tracking where your data came from.

### Checklist

- [ ] Warehouse starts and runs your queries
- [ ] Row counts differ between silver and gold, and you can explain why
- [ ] Your insight query returns a meaningful answer
- [ ] Lineage graph links gold → silver → source

> 💡 **Tip:** Notice how fast the first query is vs. subsequent ones — SQL Warehouses **cache results** and reuse them when you rerun an identical query.


# 🤖 AI-Assisted Development — Data Engineering

AI is **not a replacement** for understanding data engineering.
It is a development **assistant** that helps you write code faster, explore alternatives,
understand errors, and improve your solutions.

> ⚠️ **You are still responsible for the correctness of the result!**

---

## 1. AI Tools in Databricks

Databricks provides different AI capabilities for different purposes.
It is important to understand the distinction between them.

| Tool | Main purpose | Typical use |
|---|---|---|
| **Databricks Assistant / Genie Code** | AI coding assistant | Generate, explain, debug and improve PySpark and SQL code |
| **AI/BI Genie** | Conversational data analytics | Ask questions about business data using natural language |

> 💡 **Key takeaway:**
> Use **Genie Code** (Databricks Assistant) as your *coding partner* when developing notebooks and pipelines.
> **AI/BI Genie** is primarily designed for exploring and analyzing data through natural language.

---

## 2. How to Use AI Effectively

Follow this **five-step workflow** when working with AI:

### Step 1 — Describe the task clearly

- Explain the input data, expected output, business logic and constraints.
- Provide table schemas and sample records when relevant.

### Step 2 — Ask for an initial solution

- Request PySpark, SQL or Lakeflow Declarative Pipelines code.
- Start with **smaller tasks** instead of requesting an entire solution at once.

### Step 3 — Review the generated code

- Check joins, filters, NULL handling, data types and business logic.
- Make sure you **understand** what the code does.

### Step 4 — Execute and validate

- Run the code against **real data**.
- Check row counts, duplicates, NULL values and expected results.

### Step 5 — Iterate and improve

- Share actual **error messages** or unexpected results with AI.
- Ask it to investigate, explain and suggest improvements.

---

### Prompting tips

A good prompt should contain:

| Element | Question to ask yourself |
|---|---|
| **Context** | What are you trying to achieve? |
| **Input** | What data and schema are available? |
| **Requirements** | What transformations or business rules are needed? |
| **Expected output** | What should the result look like? |
| **Constraints** | Are there any technical or performance requirements? |

#### Example

❌ **Instead of:**

> Create a PySpark transformation.

✅ **Try:**

> I have a DataFrame containing customer records with `id`, `firstname`, `lastname` and `email` columns.
> Some customer IDs are NULL and some records are duplicated.
> Create a PySpark transformation that removes records with NULL IDs and keeps only one record per customer ID.
> Explain the logic and suggest how to validate the result.

---

## 3. Limitations — What Should You Be Aware Of?

AI-generated code is **not necessarily correct**, even if it executes without errors.

| Limitation | Why it matters |
|---|---|
| **Hallucinations** | AI can invent columns, functions or APIs. |
| **Incorrect business logic** | Code can run successfully but still produce incorrect results. |
| **Limited context** | AI may misunderstand requirements or miss important dependencies. |
| **Performance** | Working code is not necessarily efficient Spark code. |
| **Data quality** | AI cannot automatically determine what constitutes valid business data. |
| **Pipeline semantics** | AI may misunderstand streaming vs. batch processing or pipeline-specific constraints. |
| **Security** | Never share sensitive data, credentials or secrets in prompts. |

### Golden Rules

1. Never blindly accept generated code.
2. Always inspect the code before executing it.
3. Validate the output, not just whether the code runs.
4. Understand the solution well enough to explain it.
5. Consider performance and scalability.
6. Ask AI to explain its assumptions and design decisions.

---

## 4. 🧪 Practical Exercise — AI-Assisted Development

### Challenge 1: Re-do Lab 2 — ETL Challenge

**Objective:** Repeat the Lab 2 ETL challenge using Databricks Assistant / Genie Code.

This time, use AI as your **pair programmer** to help you implement the SQL-based transformations.

**Your tasks:**

1. Ask AI to help you understand the requirements.
2. Generate the SQL queries step by step.
3. Execute the queries and inspect the results.
4. Compare the AI-generated solution with your original implementation.
5. Identify at least one potential issue or improvement in the generated code.

**Bonus challenge:**

> Ask AI to review your original implementation and suggest improvements in terms of
> readability, correctness and performance.

### Challenge 2: Check on the SQL Editor assitance

---

> 💡 **Remember:** Treat AI as a *pair programmer*, not an autopilot.
> It drafts — you review, test and own the result!

# 🚀 Version Control & Deployment – Git and Databricks Asset Bundles

So far, we have developed and executed code directly in Databricks notebooks. But how do we manage our code in a real-world project with multiple developers, different environments and production deployments?

Two important concepts help us solve this problem: **Git** and **Databricks Asset Bundles (DABs)**.

---

## 1. Git – Version Control

**Git** is a version control system that tracks changes in source code and allows multiple developers to collaborate.

Instead of keeping different copies of notebooks or manually tracking changes, Git maintains the history of the project.

### Key concepts

| Concept | Description |
|---|---|
| Repository | A place where project files and their history are stored |
| Commit | A snapshot of changes with a descriptive message |
| Branch | An independent line of development |
| Merge | Combining changes from different branches |
| Pull Request (PR) | A request to review and merge changes into another branch |
| Revert | Undoing a previous change while keeping the history |

### Typical development workflow

1. Clone or open the project from a Git repository.
2. Create a feature branch for your changes.
3. Develop and test your code.
4. Commit and push your changes.
5. Create a Pull Request for code review.
6. Merge the changes into the main branch.

**Why is this important?**
- Track who changed what and when.
- Collaborate without overwriting each other's work.
- Review code before deploying it.
- Revert changes if something goes wrong.
- Maintain a history of the project.

💡 **Git in Databricks:** Databricks Git folders allow you to connect a workspace folder to a remote Git repository (e.g. GitHub, GitLab or Azure DevOps) and work with version-controlled notebooks and source files.

---

## 2. Infrastructure as Code (IaC)

Traditionally, developers and administrators create and configure resources manually through a graphical interface.

**Infrastructure as Code (IaC)** means defining infrastructure and resource configurations in code or configuration files instead of creating everything manually.

For example, instead of manually creating a Databricks Job in the UI, we can describe its configuration in a YAML file.

Benefits:
- Repeatable and consistent deployments.
- Configuration can be stored in Git.
- Changes can be reviewed and tracked.
- Resources can be recreated in different environments.
- Less manual work and fewer configuration errors.

---

## 3. Databricks Asset Bundles (DABs)

**Databricks Asset Bundles**, now officially called **Declarative Automation Bundles**, extend these principles to Databricks projects.

A bundle describes a complete deployable project, including:

- Source code (Python, SQL, notebooks).
- Databricks resources (Jobs, Lakeflow Pipelines, etc.).
- Resource configurations and dependencies.
- Environment-specific settings.
- Tests and deployment configuration.

Instead of manually creating resources in the Databricks UI, we define them as code and deploy them together.

### Example project structure

```text
my-data-project/
│
├── databricks.yml          # Main bundle configuration
│
├── src/
│   ├── bronze.py
│   ├── silver.py
│   └── gold.sql
│
├── resources/
│   └── jobs.yml            # Job definitions
│
└── tests/
    └── test_transformations.py
```

### Minimal configuration example

The `databricks.yml` file describes the project and its deployment targets:

```yaml
bundle:
  name: my-data-project

targets:
  dev:
    mode: development

  prod:
    mode: production
```

This defines two deployment targets: development and production.

The same project can be deployed to different environments without maintaining separate copies of the code.

### How does deployment work?

Typical commands using the Databricks CLI:

```bash
# Check the bundle configuration
databricks bundle validate

# Deploy to development
databricks bundle deploy -t dev

# Deploy to production
databricks bundle deploy -t prod
```

The bundle creates or updates the resources defined in the project.

---

## 4. CI/CD – Automating the Deployment Process

**CI/CD (Continuous Integration / Continuous Delivery)** automates the process of testing and deploying code.

For example, after a developer merges a change into the main branch, an automated pipeline can:

1. Validate the code and configuration.
2. Run automated tests.
3. Deploy the project to the development or staging environment.
4. Perform additional validation.
5. Deploy to production after approval.

### Putting everything together

```text
       DEVELOPER
           |
           v
    Develop in Databricks
           |
           v
       Git Commit
           |
           v
      Pull Request
           |
           v
      Code Review
           |
           v
       Git Merge
           |
           v
      CI/CD Pipeline
           |
           v
    DAB Deployment
           |
      +----+----+
      |         |
      v         v
     DEV       PROD
```

---

## 5. How Do These Concepts Fit Together?

| Technology | Responsibility |
|---|---|
| **Git** | Version control, collaboration and change history |
| **IaC** | Define infrastructure and resources as code |
| **DAB** | Define and deploy Databricks project resources consistently |
| **CI/CD** | Automate testing and deployment |

**Important distinction:** DABs manage Databricks project resources, such as Jobs and Pipelines. Larger cloud infrastructure components, such as networks, cloud resources and workspace provisioning, are typically managed separately using tools like Terraform.

---

### 💡 Key Takeaways

- Git helps teams manage and track code changes.
- IaC makes infrastructure and configuration reproducible.
- DABs allow us to define and deploy Databricks projects as a unit.
- CI/CD automates testing and deployment.
- Together, these practices help us move from notebook experimentation to maintainable, production-ready data engineering projects.

**The main idea:** We don't just deploy notebooks. We manage, version, test and deploy the entire data engineering project.